# Unit 8 (Part 2) — Doom with Sample Factory (APPO)

**Pass criterion:** `mean_reward − std_reward ≥ 5`

The agent learns to **survive on an acid floor by collecting health packs** (`doom_health_gathering_supreme`). Input: raw game frames → CNN + PPO.

**Sample Factory** is a very fast library that runs many game instances in parallel (**APPO** = asynchronous PPO: data collection and learning run at the same time without waiting for each other).

**Use a GPU:** `Runtime → Change runtime type → T4 GPU`.

**How to run:** `Runtime → Run all` (~45–75 min). The Hugging Face login reads the `HF_TOKEN` Colab secret.

## 1. Setup
Sample Factory requires an older `numpy`. To avoid clashing with Colab's own packages, training and evaluation run in **separate Python processes** (`!python ...`); the notebook kernel itself never imports it.

In [ ]:
!pip install -q sample-factory==2.1.1 vizdoom
!apt-get install -y -qq ffmpeg > /dev/null

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV = "doom_health_gathering_supreme"
REPO = f"{HF_USERNAME}/rl_course_vizdoom_{ENV}"
TRAIN_STEPS = 4_000_000

import base64
from IPython.display import HTML, display

def show_video(path, width=500):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 2. Hugging Face login (once, at the start)

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

## 3. Live training dashboard: TensorBoard
Refresh with ⟳ once training has started. Watch `reward` (should rise) and `len` (episode length = survival time).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/train_dir

## 4. Training
- `--num_workers=8 --num_envs_per_worker=4` → **32 Doom instances** in parallel.
- `--train_for_env_steps=4000000` → 4M steps.
- Doom-specific settings (CNN architecture, frame skip, reward scaling) come from Sample Factory's `sf_examples.vizdoom` module.

Watch `Avg episode reward` in the log: ~4 at the start, **10+** at the end.

In [ ]:
!python -m sf_examples.vizdoom.train_vizdoom --env={ENV} --num_workers=8 --num_envs_per_worker=4 --train_for_env_steps={TRAIN_STEPS} --experiment={ENV} --train_dir=/content/train_dir

## 5. Watch 🎮 + evaluate
Plays 10 episodes and records a video; `Avg episode reward` is printed at the end.

`TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` is required because PyTorch ≥ 2.6 loads checkpoints with `weights_only=True` by default, which rejects Sample Factory's checkpoint format.

In [ ]:
import os; os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # new PyTorch refuses old Sample Factory checkpoints otherwise
!python -m sf_examples.vizdoom.enjoy_vizdoom --env={ENV} --num_workers=1 --save_video --no_render --max_num_episodes=10 --experiment={ENV} --train_dir=/content/train_dir 2>&1 | grep -E "Avg episode reward|saved"
show_video(f"/content/train_dir/{ENV}/replay.mp4", width=480)

## 6. Push to the Hugging Face Hub
Same command with `--push_to_hub`: plays 10 episodes, writes the score to the model card and uploads the video and the model.

In [ ]:
!python -m sf_examples.vizdoom.enjoy_vizdoom --env={ENV} --num_workers=1 --save_video --no_render --max_num_episodes=10 --max_num_frames=100000 --experiment={ENV} --train_dir=/content/train_dir --push_to_hub --hf_repository={REPO} 2>&1 | grep -E "Avg episode reward|hub|Hub|error|Error"

## 7. Check
Doom models are checked in a special way: the dataset name in the model card must be `doom_health_gathering_supreme`.

In [ ]:
from huggingface_hub import HfApi, ModelCard
api = HfApi()
for m in api.list_models(author=HF_USERNAME, filter=["reinforcement-learning", "sample-factory"]):
    meta = ModelCard.load(m.id).data.to_dict()
    res = meta["model-index"][0]["results"][0]
    name = res["dataset"]["name"]
    mean, std = (float(x) for x in str(res["metrics"][0]["value"]).split("+/-"))
    ok = name == ENV and mean - std >= 5
    print(f"{'✅' if ok else '❌'} {m.id}: dataset={name}, score={mean - std:.2f} (needs >= 5)")

### If the threshold is not reached
Set `TRAIN_STEPS = 8_000_000` in the settings cell (the one with `HF_USERNAME`), re-run it and the training cell in step 4 (Sample Factory **resumes where it left off**), then repeat steps 5–7.